# Bâtis impactés par les incendies — notebook de lancement

**Demande métier** — *« Je souhaiterais savoir combien et positionner géographiquement
les bâtis (RP, RS, PNO) de nos sociétaires qui auraient effectivement été impactés
par les incendies. »*

**Livrable** — `livrables/carte_incendie_societaires.html`, un fichier autoportant
contenant carte, compteurs et tableaux, plus `contrats_impactes.csv` pour la gestion.

---

## Où est le code

Ce notebook ne contient plus de logique : il enchaîne les étapes et affiche les
diagnostics. Tout le traitement vit dans le package `incendie/analyse/`, un module
par étape :

| Module | Rôle |
|---|---|
| `config.py` | **les paramètres** — le seul fichier à ouvrir en usage courant |
| `donnees.py` | emprises des feux, requêtes SQL, extractions |
| `traitement.py` | diagnostics, segmentation, géocodage, appariement spatial |
| `resultats.py` | compteurs, tableaux, export de gestion |
| `carte.py` | carte interactive |
| `rapport.py` | livrable HTML |

Chaque module lit `config` **au moment de l'appel**, pas à l'import : réaffecter un
paramètre dans la cellule suivante suffit à le prendre en compte, sans redémarrer
le noyau.

Pour un lancement automatisé sans passer par le notebook :

```python
from analyse import config as cfg, analyser
cfg.FEUX_A_TRAITER = ("Var",)
resultat = analyser()          # renvoie tous les objets intermédiaires
```

## 1. Réglages

Les deux seuls réglages courants sont ici. Tous les autres paramètres — seuils de
distance, tables BigQuery, libellés métier, palettes — sont dans
`analyse/config.py`, commentés un par un.

In [ ]:
import sys
from pathlib import Path

# Le package vit à côté de ce notebook. On l'ajoute au chemin d'import pour que
# le notebook fonctionne aussi bien ouvert depuis `incendie/` que depuis la
# racine du dépôt.
_ici = Path.cwd()
_pkg = next((p for p in (_ici, _ici / "incendie", *_ici.parents)
             if (p / "analyse" / "__init__.py").exists()), None)
if _pkg is None:
    raise FileNotFoundError("Package `analyse` introuvable depuis " + str(_ici))
sys.path.insert(0, str(_pkg))

import pandas as pd
from analyse import config as cfg
from analyse import carte, donnees, rapport, resultats, traitement


def afficher(x):
    """Route les tableaux vers display(), le reste vers print()."""
    display(x) if isinstance(x, (pd.DataFrame, pd.Series)) else print(x)


# ═══ RÉGLAGES COURANTS ════════════════════════════════════════════════════ #
# Feux à traiter, parmi les clés de cfg.FEUX :
#   ("Gironde", "Biscarrosse")  → les deux feux de juillet
#   ("Var",)                    → Pontevès seul   (la virgule est obligatoire)
#   None                        → tous les feux du catalogue
cfg.FEUX_A_TRAITER = ("Gironde", "Biscarrosse")

# Compter comme impactés les contrats situés dans l'emprise mais à l'écart de
# tout bâti relevé. À activer quand le périmètre est une trace de brûlé précise —
# et c'est la seule façon de traiter un feu dont la couche de bâtiments manque.
cfg.INTEGRER_PERIMETRE = False
# ═════════════════════════════════════════════════════════════════════════ #

print("Catalogue :", ", ".join(cfg.FEUX))
print("Racine projet :", cfg.RACINE)

## 2. Emprises des feux

Les shapefiles sont ramenés en Lambert 93, un CRS manquant est réparé, et les
contours issus de vectorisation raster — celui du Var arrive en 97 polygones —
sont fusionnés en une emprise.

Le contrôle de recouvrement vérifie que les bâtiments relevés tombent bien dans
leur propre périmètre : un taux nettement inférieur à 95 % signalerait un
décalage de projection ou un mauvais appariement de fichiers.

In [ ]:
contours, batis = donnees.charger_feux(afficher)
donnees.controle_recouvrement(contours, batis, afficher)

display(contours.drop(columns="geometry"))
if len(batis):
    display(batis.groupby("feu").agg(nb_batis=("bat_id", "size"),
                                     surface_totale_m2=("surface_bati_m2", "sum"),
                                     surface_mediane_m2=("surface_bati_m2", "median"),
                                     surface_max_m2=("surface_bati_m2", "max")).round(0))

## 3. Extraction des contrats

La requête part de `contrat_mgar` et non de la table de géocodage : c'est le
**contrat le plus récent** par couple `(id_societaire, numero_intercalaire)` qui
décrit le risque assuré aujourd'hui, donc lui qui fournit l'adresse et pilote la
jointure.

Le pré-filtre sur l'emprise des feux évite de rapatrier la France entière.

In [ ]:
bbox = donnees.emprise_requete(contours, afficher)
requete = donnees.requete_contrats(bbox)
print(requete)

In [ ]:
contrats, mode_source = donnees.charger_contrats(requete)
print(f"Mode = {mode_source.upper()} — {len(contrats):,} lignes chargées".replace(",", " "))
print(f"{contrats['id'].nunique():,} clés (sociétaire, intercalaire) distinctes\n"
      .replace(",", " "))
display(contrats.head())

### Diagnostic des doublons

La requête sort en `SELECT DISTINCT` : deux lignes de même clé diffèrent donc
forcément sur au moins une colonne sélectionnée. La cause ne se devine pas, et ses
conséquences vont du bénin — même adresse géocodée deux fois — à l'anomalie
d'historisation.

L'adresse est affichée même quand elle ne varie pas : c'est elle qui rend le relevé
exploitable auprès du producteur de la donnée.

In [ ]:
traitement.diagnostiquer_doublons(contrats, afficher)

## 4. Segmentation et sinistres déclarés

Le segment se lit sur `code_sous_type` : `1`–`5` → RP, `6` → PNO, `7` → RS. Les
segments hors demande — jeune, étudiant, hébergé — sont comptés à part plutôt
qu'écartés en silence.

Le croisement avec les **sinistres incendie déclarés** est le seul contrôle externe
de la méthode : il la mesure au lieu de la supposer juste.

In [ ]:
contrats = traitement.segmenter(contrats, afficher)

In [ ]:
requete_sin = donnees.requete_sinistres()
print(requete_sin)

sinistres, mode_sinistres = donnees.charger_sinistres(requete_sin, mode_source)
contrats = traitement.rattacher_sinistres(contrats, sinistres, afficher)

## 5. Qualité du géocodage

Une distance ne vaut que ce que vaut la position dont elle part. La colonne
`level_contrat_mgar` reprend les niveaux de la Base Adresse Nationale :

| Modalité | Point posé sur | Écart typique | Exploitable ? |
|---|---|---|---|
| `housenumber` | le point adresse | quelques mètres | **oui** |
| `street` | l'axe de la voie | 10 à 50 m | oui, plafonné |
| `locality` | le centre d'un lieu-dit | centaines de mètres | **non** |
| `municipality` | le centroïde de la commune | kilomètres | **non** |

Une heuristique historique — plusieurs voies sur un même point trahissent un
centroïde — est confrontée à cette colonne. Elle s'est révélée fausse à 94 % sur
les données réelles : les libellés incriminés étaient `EMPLACEMENT 667`,
`KHELUS LOT 320`, `PAVILLON 901`, soit des **campings et villages de vacances** où
chaque « voie » est un numéro d'emplacement.

In [ ]:
points, colonne_precision = traitement.geolocaliser(contrats, afficher)

## 6. Appariement spatial

| Niveau | Règle |
|---|---|
| **Certain** | dans l'emprise d'un bâti relevé, ou à moins de 5 m |
| **Très probable** | 5 à 15 m |
| **Probable** | 15 à 30 m |
| **Exposé** | dans le périmètre, au-delà de 30 m de tout bâti |
| Hors périmètre | reste |

Le géocodage borne la conclusion : les points au centroïde d'une commune ou d'un
lieu-dit sortent du comptage, les points posés sur l'axe d'une voie sont plafonnés
à « très probable ».

Un contrat portant plusieurs géocodages est ramené à un seul point : le plus
précis, puis le plus central parmi les candidats de même niveau. Ces deux critères
sont **neutres** — départager sur la distance au bâti retiendrait toujours la
position la plus incriminante et surestimerait l'impact par construction.

In [ ]:
appariement = traitement.apparier(points, batis, contours, colonne_precision, afficher)

## 7. Réponse chiffrée

« Combien ? » se décline en trois compteurs, et il faut les trois : bâtiments
distincts, contrats, sociétaires. Un immeuble ne compte qu'une fois en bâtiments
mais porte plusieurs contrats.

In [ ]:
compteurs = resultats.compteurs(appariement)
for cle, valeur in compteurs.items():
    print(f"{cle:>26} : {valeur:>6,}".replace(",", " "))

In [ ]:
display(resultats.synthese_par_feu(appariement))
display(resultats.detail_par_niveau(appariement))

In [ ]:
# Le seuil est le principal arbitrage de la méthode : mieux vaut montrer sa
# sensibilité que défendre un nombre unique.
display(resultats.sensibilite(appariement))

In [ ]:
# Matrice de validation : ce que la géométrie retient, confronté aux déclarations.
display(resultats.matrice_validation(appariement))

print(f"\nParmi les {compteurs['contrats']} contrats retenus comme impactés, "
      f"{compteurs['declares']} ont un sinistre ouvert et "
      f"{compteurs['impactes_sans_declaration']} n'ont pas déclaré à ce jour.")
print(f"{compteurs['declares_non_detectes']} contrat(s) déclarent un sinistre sans "
      f"être retenus par la géométrie : autant de cas où la méthode passe à côté.")

for tableau in (resultats.croisement_statut(appariement),
                resultats.croisement_type_bien(appariement)):
    if tableau is not None:
        display(tableau)

In [ ]:
export = resultats.exporter(appariement)
print(f"{len(export)} contrats exportés → {cfg.FICHIER_CSV}")
display(export.head(10))

## 8. Carte et livrable

Quatre variables visuelles, une seule colorée : la **couleur** porte le segment, la
**forme** le sinistre déclaré (triangle) ou non (disque), le **plein ou l'anneau** le
statut d'occupation, la **taille** le niveau de certitude.

Leaflet et jQuery sont injectés en dur depuis `incendie/assets/` : le livrable ne
dépend d'aucun CDN — seules les tuiles du fond de carte demandent le réseau.

In [ ]:
m = carte.construire(appariement, batis, contours, bbox, afficher)
carte_html = carte.rendre_autonome(m, afficher)
print(f"Carte rendue — {len(carte_html) / 1024:.0f} Ko")
m

In [ ]:
livrable = rapport.ecrire(appariement, batis, carte_html,
                          mode_source, colonne_precision, afficher=afficher)

## 9. Requêtes de contrôle

À passer une fois dans BigQuery, avant de communiquer des chiffres. Sans elles on
conclurait sur des résultats dont on ignore la fiabilité.

| Clé | Ce qu'elle établit |
|---|---|
| `jointure` | combien de contrats la jointure stricte sur l'adresse fait tomber |
| `unicite_gps` | à quel niveau de clé la table de géocodage devient unique |
| `causes_doublons_gps` | ce qui varie exactement sur les clés en doublon |
| `modalites_precision` | les modalités réelles de `level_contrat_mgar` |
| `codes_sinistre` | que `05` domine bien les survenances postérieures au feu |

In [ ]:
for nom, sql in donnees.requetes_controle(bbox).items():
    print(f"\n{'═' * 76}\n{nom}\n{'═' * 76}{sql}")

---

## Modifier l'analyse

| Ce qu'on veut changer | Où |
|---|---|
| Feux traités, intégration du périmètre | cellule 1 de ce notebook |
| Seuils de distance, niveaux d'impact | `config.py` — les libellés en dérivent |
| Tables BigQuery, colonnes extraites | `config.py`, puis `donnees.py` pour le SQL |
| Règles de segmentation, libellés métier | `config.py` |
| Traitement des doublons, appariement | `traitement.py` |
| Apparence de la carte | `carte.py` |
| Structure du rapport | `rapport.py` |

## Ajouter un feu

Déposer le dossier dans `data_incendie/`, ajouter une entrée à `cfg.FEUX` avec son
motif de fichiers, sa date de départ et sa date de relevé, puis l'ajouter à
`FEUX_A_TRAITER`. Le reste suit : chargement, appariement, compteurs, carte, HTML.

Sans couche de bâtiments, l'exécution s'arrête sur un message explicite —
`INTEGRER_PERIMETRE = True` permet alors de traiter le feu au seul périmètre.